# COB107 — Convolutional Neural Networks on MNIST (Solved)

In this lab, you will build and experiment with Convolutional Neural Networks (CNNs) for classifying handwritten digits from the MNIST dataset. You will progressively improve your architecture by adding pooling layers, regularisation, and experimenting with hyperparameters.

In [4]:
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [5]:
device = 'cpu'
batch_size = 64

### **Setup**: Download the MNIST dataset and create DataLoaders for training and testing.

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### **Task 1**: Build a CNN **without** max-pooling

Define a CNN class with **3 convolutional layers** using `3×3` filters. Use the following number of filters per layer:

- Conv Layer 1: 16 filters
- Conv Layer 2: 32 filters
- Conv Layer 3: 64 filters

After the convolutional layers, flatten the output and add a fully connected layer with 128 neurons followed by an output layer with 10 neurons.

Use `ReLU` activation after each convolutional and the first fully connected layer. Use `padding=1` to keep the spatial dimensions unchanged through convolutions.

**Do not use any pooling layers in this task.**

*Hint: MNIST images are 1×28×28. With padding=1 and 3×3 filters, each convolution preserves the 28×28 spatial size. After 3 such layers the feature maps will be 64×28×28, so your first Linear layer should accept 64 × 28 × 28 = 50176 inputs.*

In [ ]:
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        INPUT_NEURONS = 28*28
        self.conv1 = nn.Conv2d(in_channels=INPUT_NEURONS, out_channels=INPUT_NEURONS*16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=INPUT_NEURONS*16, out_channels=INPUT_NEURONS*32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=INPUT_NEURONS*32, out_channels=INPUT_NEURONS*64, kernel_size=3, padding=1)
        self.linear1 = nn.Linear(in_features=INPUT_NEURONS*64, out_features=128)
        self.output_layer = nn.Linear(in_features=128, out_features=10)


    def forward(self, x: torch.Tensor):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = torch.flatten(x, 1)
        x = F.relu(self.linear1(x))
        x = self.output_layer(x)
        return x


net = Net()

Write `train` and `evaluate` functions, then train your CNN for **5 epochs** using `CrossEntropyLoss` and `SGD` with a learning rate of `0.01`. Report the loss and test accuracy after each epoch.

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

for epoch in range(2):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

### **Task 2**: Add Max-Pooling layers to your CNN

Copy your CNN from Task 1 and add a `MaxPool2d(2, 2)` layer after the first and second convolutional layers.

The architecture should now be:

- Conv1 (16 filters, 3×3, padding=1) → ReLU → **MaxPool(2×2)**
- Conv2 (32 filters, 3×3, padding=1) → ReLU → **MaxPool(2×2)**
- Conv3 (64 filters, 3×3, padding=1) → ReLU
- Flatten → FC(128) → ReLU → FC(10)

Train this model for **5 epochs** with the same settings as Task 1. Compare the test accuracy and note the reduction in the number of parameters.

*Hint: After Conv1 + Pool, the spatial size becomes 14×14. After Conv2 + Pool, it becomes 7×7. So the flattened size before the FC layer is 64 × 7 × 7 = 3136.*

### **Task 3**: Introduce regularisation through the optimiser

Using the CNN from Task 2, apply **L2 regularisation** (weight decay) by passing the `weight_decay` parameter to your optimiser.

Train the model for **5 epochs** using `SGD` with `lr=0.01` and `weight_decay=1e-2`.

Compare the test accuracy with your Task 2 results. Does regularisation help?

### **Task 4**: Experiment with architecture parameters

In this task, you will explore how different architecture choices affect the **dimensions of activation maps** (feature maps) as data passes through the network.

Build a small utility function `print_activation_shapes(model, input_size)` that takes a model and an input tensor, passes it through each layer one at a time, and prints the output shape after every layer.

Then, experiment with at least **two** of the following changes and observe how the activation map dimensions change:

1. Change the kernel size from `3×3` to `5×5` (with `padding=2` to preserve dimensions, or `padding=0` to shrink them).
2. Add a third `MaxPool2d` layer after Conv3.
3. Increase the number of filters to 32 → 64 → 128.
4. Use a `stride=2` in one of the convolutional layers instead of max-pooling.

For each experiment, print the activation shapes.